# Data cleaning

This notebook prepares the HCV dataset for the later EDA, statistical and machine learning stages. It examines missing values, compares imputation strategies under cross-validation, removes the suspected blood donors, imputes the remaining gaps with the median, reports (but does not remove) outliers, and saves the cleaned file.

## Imports

In [1]:
# Standard library
from pathlib import Path

# Third-party libraries
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import MonteCarloMethod, chi2_contingency
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Configuration

Paths, the random seed and the column lists are defined once here. The notebook is run from the project folder, so paths are relative.

In [2]:
# Input and output files, relative to the project folder.
DATA_PATH = Path("hcvdat0.csv")
OUTPUT_PATH = Path("hcv_median_imputed.csv")

# One seed is used everywhere so the results can be reproduced exactly.
SEED = 42
ALPHA = 0.05          # significance level for the missingness tests
N_RESAMPLES = 10000   # Monte Carlo resamples per test
N_FOLDS = 5           # cross-validation folds

# Order used for every table. "Suspected Blood Donor" is dropped later.
CLASS_ORDER = ["Healthy", "Hepatitis", "Fibrosis", "Cirrhosis"]

# The 11 measured variables (Sex is categorical, so it is kept separate).
NUMERIC_COLUMNS = [
    "Age", "ALB", "ALP", "ALT", "AST", "BIL", "CHE", "CHOL",
    "CREA", "GGT", "PROT",
]

# Numeric code for each cleaned label (kept from the original dataset).
CATEGORY_CODES = {
    "Healthy": 0,
    "Suspected Blood Donor": 1,
    "Hepatitis": 2,
    "Fibrosis": 3,
    "Cirrhosis": 4,
}

## Load the raw data

In [3]:
# Read the raw file exactly as supplied and look at the first rows.
raw_df = pd.read_csv(DATA_PATH)
raw_df.head()

,Unnamed: 0,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT
0,1,0=Blood Donor,32,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0
1,2,0=Blood Donor,32,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5
2,3,0=Blood Donor,32,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3
3,4,0=Blood Donor,32,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7
4,5,0=Blood Donor,32,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7


## Tidy the labels

The first column is only a row index, so it is dropped. Category labels such as `0=Blood Donor` are reduced to the text after the equals sign, and "Blood Donor" is renamed "Healthy". `cat_num` stores the numeric code of each label; it is a label code, not a measurement.

In [4]:
# The first column is only a row number, so it is dropped.
df = raw_df.drop(columns=["Unnamed: 0"])

# Labels look like "0=Blood Donor"; keep only the text after the "=".
df["Category"] = df["Category"].str.replace(r"^[^=]*=", "", regex=True)

# Use the clearer names "Healthy" and "Suspected Blood Donor".
df["Category"] = df["Category"].replace({
    "Blood Donor": "Healthy",
    "suspect Blood Donor": "Suspected Blood Donor",
})

# Add the numeric label code (a label, not a measurement).
df["cat_num"] = df["Category"].map(CATEGORY_CODES).astype("Int64")

print(f"Rows: {df.shape[0]}, columns: {df.shape[1]}")
df.head()

Rows: 615, columns: 14


,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT,cat_num
0,Healthy,32,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0,0
1,Healthy,32,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5,0
2,Healthy,32,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3,0
3,Healthy,32,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7,0
4,Healthy,32,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7,0


## Class sizes

The classes are strongly imbalanced, which matters for every later step.

In [5]:
# Count the patients in each category.
df.groupby(["cat_num", "Category"]).size().rename("Count").reset_index()

,cat_num,Category,Count
0,0,Healthy,533
1,1,Suspected Blood Donor,7
2,2,Hepatitis,24
3,3,Fibrosis,21
4,4,Cirrhosis,30


## Missing values

Number and percentage of missing values in each column.

In [6]:
# Count missing values per column and express them as a percentage.
missing_summary = pd.DataFrame({
    "Missing": df.isna().sum(),
    "Missing (%)": df.isna().mean().mul(100).round(2),
})
missing_summary

,Missing,Missing (%)
Category,0,0.00
Age,0,0.00
Sex,0,0.00
ALB,1,0.16
ALP,18,2.93
ALT,1,0.16
AST,0,0.00
BIL,0,0.00
CHE,0,0.00
CHOL,10,1.63


### Missing values by category

Percentage of each column that is missing within each diagnostic group. If missingness differed between groups, simply deleting incomplete rows would bias the sample.

In [7]:
# Keep only the columns that actually have missing values.
missing_columns = missing_summary.index[missing_summary["Missing"] > 0]

# Mark each value as missing (True) or present (False), then average
# within each category to get the percentage missing.
missing_by_category = (
    df[missing_columns].isna().groupby(df["Category"]).mean().mul(100)
)
missing_by_category.round(2)

,ALB,ALP,ALT,CHOL,PROT
Category,,,,,
Cirrhosis,3.33,20.00,0.00,6.67,3.33
Fibrosis,0.00,42.86,0.00,4.76,0.00
Healthy,0.00,0.00,0.00,1.31,0.00
Hepatitis,0.00,12.50,4.17,0.00,0.00
Suspected Blood Donor,0.00,0.00,0.00,0.00,0.00


## Is missingness related to diagnosis?

For each incomplete column a chi-square test compares the number of missing and observed values across categories. Because some cells have very small counts, the p-value is estimated by Monte Carlo resampling (10,000 resamples, fixed seed) rather than by the large-sample approximation. A Bonferroni correction adjusts for testing several columns. A non-significant result does not prove that values are missing at random.

In [8]:
# Test each incomplete column separately.
p_values = []
for column in missing_columns:
    # Count missing and present values within each category.
    counts = pd.crosstab(df["Category"], df[column].isna())

    # Monte Carlo sampling is used because some counts are very small.
    # The seed is restarted for every column so each test is repeatable.
    method = MonteCarloMethod(
        n_resamples=N_RESAMPLES, rng=np.random.default_rng(SEED)
    )
    result = chi2_contingency(counts, correction=False, method=method)
    p_values.append(result.pvalue)

missingness = pd.DataFrame({
    "Column": missing_columns,
    "Missing": df[missing_columns].isna().sum().to_numpy(),
    "p_value": p_values,
})

# Bonferroni correction: multiply by the number of tests (maximum of 1).
missingness["adjusted_p"] = (
    missingness["p_value"] * len(missingness)
).clip(upper=1)
missingness["Significant"] = missingness["adjusted_p"] < ALPHA
missingness

,Column,Missing,p_value,adjusted_p,Significant
0,ALB,1,0.130987,0.654935,False
1,ALP,18,0.000100,0.000500,True
2,ALT,1,0.086191,0.430957,False
3,CHOL,10,0.222378,1.000000,False
4,PROT,1,0.130987,0.654935,False


### Findings: missing values

Five columns contain missing values: ALB (1), ALP (18), ALT (1), CHOL (10) and PROT (1). After Bonferroni adjustment for five tests, only ALP shows an association with diagnosis (adjusted p = 0.0005); the other four are not significant (adjusted p between 0.43 and 1.00).

ALP is missing for 42.86% of fibrosis patients (9 of 21), 20.00% of cirrhosis patients (6 of 30) and 12.50% of hepatitis patients (3 of 24), but for none of the healthy or suspected donors. Missingness is therefore concentrated in liver disease, which is evidence against ALP being missing completely at random. Deleting incomplete rows would remove a disproportionate share of the minority-class patients, so imputation is preferred to deletion.

The non-significant results for the other columns do not show that those values are missing at random, and ALB, ALT and PROT have a single missing value each, so the tests have little power. These tests examine only the diagnosis and cannot distinguish missing at random from missing not at random. With 10,000 resamples the smallest attainable raw p-value is about 0.0001, which ALP reaches; its adjusted value reflects the resolution of the simulation, not a precise estimate.

## Comparing imputation methods

Choosing an imputation method because it looks plausible is not enough, so three methods are compared on identical cross-validation folds: median, median with missing-value flags, and iterative (regression-based) imputation. Each is paired with Logistic Regression and Random Forest. Macro F1, balanced accuracy and per-class recall are reported together, because no single metric is sufficient with such small minority classes.

The comparison uses the 608 patients remaining after the seven suspected blood donors are removed. The dataframe `df` itself is not modified.

### Four-class dataset

In [9]:
# Remove the seven suspected donors; the rest of this section uses
# the remaining 608 patients. The full dataframe df is left unchanged.
comparison_df = df[df["Category"] != "Suspected Blood Donor"]
comparison_df = comparison_df.reset_index(drop=True)

# Predictors exclude Category and cat_num because they encode the outcome.
features = comparison_df[NUMERIC_COLUMNS + ["Sex"]]
target = comparison_df["Category"]

# Class sizes after removing the suspected donors.
target.value_counts().reindex(CLASS_ORDER).rename("Patients")

Category
Healthy      533
Hepatitis     24
Fibrosis      21
Cirrhosis     30
Name: Patients, dtype: int64

### Missing values in the retained patients

In [10]:
# Mark missing numeric values and count them within each category.
missing_flags = features[NUMERIC_COLUMNS].isna()
missing_counts = missing_flags.groupby(target).sum().reindex(CLASS_ORDER)

# Show only the columns that have at least one missing value.
missing_counts.loc[:, missing_flags.any()]

,ALB,ALP,ALT,CHOL,PROT
Category,,,,,
Healthy,0,0,0,7,0
Hepatitis,0,3,1,0,0
Fibrosis,0,9,0,1,0
Cirrhosis,1,6,0,2,1


### Folds, imputers and models

The stratified folds are created once and reused for every method, so all methods are tested on the same patients. Class weights are balanced for both models; there is no oversampling or tuning. Iterative imputation predicts each missing value from the other numeric variables. It is single imputation, not pooled multiple imputation.

In [11]:
# Create the five folds once. Every method below reuses these exact
# splits, so all methods are tested on the same patients.
splits = list(
    StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    .split(features, target)
)

# The three ways of filling missing values that are compared.
imputers = {
    "Median": SimpleImputer(strategy="median"),
    "Median + flags": SimpleImputer(strategy="median", add_indicator=True),
    "Iterative": IterativeImputer(
        initial_strategy="median", max_iter=50, tol=0.001,
        min_value=0, random_state=SEED,
    ),
}

# The two classifiers. Class weights are balanced because the
# disease groups are small.
models = {
    "Logistic Regression": LogisticRegression(
        C=1, class_weight="balanced", max_iter=5000, random_state=SEED,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, max_features="sqrt",
        class_weight="balanced", random_state=SEED, n_jobs=-1,
    ),
}

# Names of the scores that are calculated for every run.
METRICS = ["Macro F1", "Balanced accuracy"]
METRICS += [name + " recall" for name in CLASS_ORDER]

### Run the comparison

The imputer, scaler and Sex encoder sit inside the pipeline, so medians and regression models are learned from the training folds only. Every patient is predicted by a model that did not see that patient.

In [12]:
score_rows = []
predictions_by_run = {}

for model_name, model in models.items():
    for method_name, imputer in imputers.items():
        # Preprocessing is fitted on the training folds only: fill missing
        # values, scale the numbers and encode Sex, then fit the model.
        numeric_steps = Pipeline([
            ("imputer", clone(imputer)),
            ("scaler", StandardScaler()),
        ])
        preprocessing = ColumnTransformer([
            ("numeric", numeric_steps, NUMERIC_COLUMNS),
            ("sex", OneHotEncoder(handle_unknown="ignore"), ["Sex"]),
        ])
        pipeline = Pipeline([
            ("preprocessing", preprocessing),
            ("model", clone(model)),
        ])

        # Out-of-fold predictions: each patient is predicted by a model
        # that never saw that patient during training.
        predictions = pd.Series(index=target.index, dtype=object)

        for fold, (train, test) in enumerate(splits, start=1):
            fitted = clone(pipeline).fit(
                features.iloc[train], target.iloc[train]
            )
            predicted = fitted.predict(features.iloc[test])
            actual = target.iloc[test]
            predictions.iloc[test] = predicted

            # Scores for this fold.
            scores = {
                "Model": model_name,
                "Imputation": method_name,
                "Fold": fold,
                "Macro F1": f1_score(actual, predicted, average="macro"),
                "Balanced accuracy": balanced_accuracy_score(
                    actual, predicted
                ),
            }
            # Recall of each diagnosis separately (share correctly found).
            recalls = recall_score(
                actual, predicted, labels=CLASS_ORDER, average=None,
                zero_division=0,
            )
            for name, value in zip(CLASS_ORDER, recalls):
                scores[name + " recall"] = value
            score_rows.append(scores)

        predictions_by_run[(model_name, method_name)] = predictions

results = pd.DataFrame(score_rows)

### Mean scores and fold variation

Each cell is the mean and sample standard deviation over the five folds. The standard deviation is not a confidence interval, because the training sets overlap. Each fold holds only four or five hepatitis and fibrosis patients, so one error moves minority-class recall considerably.

In [13]:
# Mean and standard deviation over the five folds for each combination.
summary = results.groupby(["Model", "Imputation"])[METRICS].agg(
    ["mean", "std"]
)

# Combine them into one readable "mean ± sd" text per cell.
comparison = pd.DataFrame(index=summary.index)
for metric in METRICS:
    comparison[metric] = (
        summary[(metric, "mean")].map("{:.3f}".format)
        + " ± "
        + summary[(metric, "std")].map("{:.3f}".format)
    )

# Overall scores first, then recall for each diagnosis.
display(comparison[["Macro F1", "Balanced accuracy"]])
display(comparison[[name + " recall" for name in CLASS_ORDER]])

Macro F1 Balanced accuracy
Model               Imputation                                     
Logistic Regression Iterative       0.600 ± 0.058     0.678 ± 0.066
                    Median          0.612 ± 0.061     0.692 ± 0.057
                    Median + flags  0.647 ± 0.020     0.686 ± 0.035
Random Forest       Iterative       0.662 ± 0.128     0.611 ± 0.129
                    Median          0.640 ± 0.149     0.590 ± 0.152
                    Median + flags  0.660 ± 0.155     0.619 ± 0.161

Healthy recall Hepatitis recall  \
Model               Imputation                                       
Logistic Regression Iterative       0.919 ± 0.055    0.680 ± 0.228   
                    Median          0.931 ± 0.040    0.640 ± 0.261   
                    Median + flags  0.974 ± 0.008    0.640 ± 0.297   
Random Forest       Iterative       1.000 ± 0.000    0.370 ± 0.254   
                    Median          1.000 ± 0.000    0.320 ± 0.303   
                    Median + flags  1.000 ± 0.000    0.320 ± 0.268   

                                   Fibrosis recall Cirrhosis recall  
Model               Imputation                                       
Logistic Regression Iterative        0.480 ± 0.254    0.633 ± 0.274  
                    Median           0.530 ± 0.220    0.667 ± 0.312  
                    Median + flags   0.430 ± 0.208    0.700 ± 0.274  
Random Forest       Iterative        0.340 ± 0.290    0.733 ± 0.190  
                    Median           0.340 ± 0.290    0.700 ± 0.217  
                    Median + flags   0.390 ± 0.342    0.767 ± 0.190

### Difference from median imputation

Positive values mean the alternative scored higher than median on the same folds; negative values favour median. This shows trade-offs between metrics rather than a single winner.

In [14]:
# Mean score of every model and imputation method across the folds.
means = results.groupby(["Model", "Imputation"])[METRICS].mean()

# Subtract the median-imputation scores of the same model.
rows = []
for model_name in models:
    median_means = means.loc[(model_name, "Median")]
    for method_name in ["Median + flags", "Iterative"]:
        change = means.loc[(model_name, method_name)] - median_means
        rows.append(change.rename((model_name, method_name)))

differences = pd.DataFrame(rows)
differences.index.names = ["Model", "Compared with median"]
differences.round(3)

Macro F1  Balanced accuracy  \
Model               Compared with median                                
Logistic Regression Median + flags           0.035             -0.006   
                    Iterative               -0.012             -0.014   
Random Forest       Median + flags           0.019              0.029   
                    Iterative                0.022              0.021   

                                          Healthy recall  Hepatitis recall  \
Model               Compared with median                                     
Logistic Regression Median + flags                 0.043              0.00   
                    Iterative                     -0.011              0.04   
Random Forest       Median + flags                 0.000              0.00   
                    Iterative                      0.000              0.05   

                                          Fibrosis recall  Cirrhosis recall  
Model               Compared with median                                     
Logistic Regression Median + flags                  -0.10             0.033  
                    Iterative                       -0.05            -0.033  
Random Forest       Median + flags                   0.05             0.067  
                    Iterative                        0.00             0.033

### Held-out mistakes

Pooled confusion matrices: each patient is counted once per model and method. Rows are the actual diagnosis and columns the prediction.

In [15]:
# One confusion matrix per model and imputation method.
for (model_name, method_name), predictions in predictions_by_run.items():
    # Rows are the actual diagnosis, columns the predicted one.
    confusion = pd.crosstab(target, predictions).reindex(
        index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0
    )
    print(f"{model_name} - {method_name}")
    display(confusion)

Logistic Regression - Median


col_0,Healthy,Hepatitis,Fibrosis,Cirrhosis
Category,,,,
Healthy,496,20,10,7
Hepatitis,2,15,5,2
Fibrosis,1,7,11,2
Cirrhosis,1,1,8,20


Logistic Regression - Median + flags


col_0,Healthy,Hepatitis,Fibrosis,Cirrhosis
Category,,,,
Healthy,519,5,6,3
Hepatitis,2,15,5,2
Fibrosis,2,7,9,3
Cirrhosis,1,1,7,21


Logistic Regression - Iterative


col_0,Healthy,Hepatitis,Fibrosis,Cirrhosis
Category,,,,
Healthy,490,21,15,7
Hepatitis,2,16,3,3
Fibrosis,2,6,10,3
Cirrhosis,2,3,6,19


... (2 outputs left out)

Random Forest - Median + flags


col_0,Healthy,Hepatitis,Fibrosis,Cirrhosis
Category,,,,
Healthy,533,0,0,0
Hepatitis,11,8,4,1
Fibrosis,7,4,8,2
Cirrhosis,3,1,3,23


Random Forest - Iterative


col_0,Healthy,Hepatitis,Fibrosis,Cirrhosis
Category,,,,
Healthy,533,0,0,0
Hepatitis,11,9,2,2
Fibrosis,7,5,7,2
Cirrhosis,6,0,2,22


### Limits of the comparison

Median imputation can be justified as a simple baseline, but not as the best method for every objective: a method may raise macro F1 while lowering fibrosis recall, and choosing the main metric after seeing the scores would bias the conclusion.

ALP is missing mainly in the disease groups, so a missing-value flag may encode how tests were ordered rather than anything biological. Performance could change in a clinic that measures ALP routinely. Median imputation also does not correct for data that are missing not at random.

One five-fold split with fixed model settings cannot establish statistical superiority, performance on new patients or the accuracy of the imputed values. Two recent studies support the design of the comparison without deciding which method wins here:

- [Sisk et al. (2023)](https://doi.org/10.1177/09622802231165001): the imputation strategy should match how the model will be deployed, and missing indicators are informative only under particular missingness patterns.
- [Shadbahr et al. (2023)](https://doi.org/10.1038/s43856-023-00356-z): no imputation method gave the best classifier across the datasets studied.

### Findings: imputation comparison

No method is best on every metric. With Logistic Regression, median imputation gives macro F1 0.612, balanced accuracy 0.692 and fibrosis recall 0.530. Median with flags gives 0.647, 0.686 and 0.430, and iterative imputation gives 0.600, 0.678 and 0.480. Random Forest with iterative imputation has the highest macro F1 (0.662), but its fibrosis recall is only 0.340, and Random Forest recalls only 8 to 9 of 24 hepatitis patients under every method.

Median imputation with Logistic Regression has the highest balanced accuracy and the highest fibrosis recall of the six configurations. Differences between methods are small compared with the fold-to-fold standard deviations (typically 0.05 to 0.15 for macro F1), so the evidence supports median imputation as a simple, adequate choice, not as a demonstrably superior one. Because the same folds were used to compare and to select, the decision is provisional and has no independent validation.

**Decision:** impute the retained patients with the median.

## Apply median imputation

The 608 retained patients are imputed with the column median. Diagnosis and Sex are not changed. The imputer is fitted on the full cohort only after the cross-validation above is finished. Any later model validation should start from the unimputed data and fit the imputer within each training fold, so `df_imputed` should not itself be cross-validated.

In [16]:
# Fill the gaps in the numeric columns with each column's median.
# Category and Sex are not changed.
df_imputed = comparison_df.copy()
df_imputed[NUMERIC_COLUMNS] = SimpleImputer(strategy="median").fit_transform(
    comparison_df[NUMERIC_COLUMNS]
)

# Every count should now be zero.
df_imputed.isna().sum().rename("Missing after imputation")

Category    0
Age         0
Sex         0
ALB         0
ALP         0
ALT         0
AST         0
BIL         0
CHE         0
CHOL        0
CREA        0
GGT         0
PROT        0
cat_num     0
Name: Missing after imputation, dtype: int64

## Outlier detection (report only)

Outliers are identified in two ways, both on the imputed numeric variables (Sex is excluded). The first is the interquartile range rule: a value is an outlier if it lies more than 1.5 times the IQR below the first quartile or above the third quartile, assessed column by column. The second is an Isolation Forest (100 trees, 256 samples per tree, contamination 0.05), which flags rows that are unusual across all eleven variables jointly. The quartiles are taken over all 608 patients. Nothing is removed in this section.

In [17]:
# Outliers are judged on the imputed measurements only (no Sex).
numeric_data = df_imputed[NUMERIC_COLUMNS]

# The IQR is the spread of the middle half of the values.
q1 = numeric_data.quantile(0.25)
q3 = numeric_data.quantile(0.75)
iqr = q3 - q1

# A value is an outlier if it is more than 1.5 IQR below Q1 or above Q3.
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
iqr_outliers = (numeric_data < lower) | (numeric_data > upper)

# Number of outlier values in each column.
iqr_outliers.sum().rename("IQR outlier values")

Age      1
ALB     22
ALP     12
ALT     34
AST     61
BIL     46
CHE     23
CHOL    12
CREA    10
GGT     68
PROT    15
Name: IQR outlier values, dtype: int64

In [18]:
# Isolation Forest looks at all 11 variables together and flags the
# 5% of rows that are easiest to separate from the rest.
forest = IsolationForest(
    n_estimators=100, max_samples=256, contamination=0.05,
    random_state=SEED,
)

# fit_predict returns -1 for an outlier and 1 for a normal row.
forest_outliers = pd.Series(
    forest.fit_predict(numeric_data) == -1, index=df_imputed.index
)

print("Rows flagged by Isolation Forest:", forest_outliers.sum())

Rows flagged by Isolation Forest: 31


### Flagged rows by category

The number of rows in each category with at least one IQR outlier, and the number flagged by the Isolation Forest, with each as a percentage of the category size.

In [19]:
# Count flagged rows in each category: rows with at least one IQR
# outlier, and rows flagged by the Isolation Forest.
flagged = pd.DataFrame({
    "Patients": df_imputed["Category"].value_counts(),
    "IQR rows": iqr_outliers.any(axis=1).groupby(df_imputed["Category"]).sum(),
    "Isolation Forest rows": forest_outliers.groupby(
        df_imputed["Category"]
    ).sum(),
}).reindex(CLASS_ORDER)

# Express each count as a percentage of the category size.
flagged["IQR (%)"] = (100 * flagged["IQR rows"] / flagged["Patients"]).round(1)
flagged["Isolation Forest (%)"] = (
    100 * flagged["Isolation Forest rows"] / flagged["Patients"]
).round(1)

flagged

,Patients,IQR rows,Isolation Forest rows,IQR (%),Isolation Forest (%)
Category,,,,,
Healthy,533,104,3,19.5,0.6
Hepatitis,24,15,4,62.5,16.7
Fibrosis,21,17,2,81.0,9.5
Cirrhosis,30,30,22,100.0,73.3


### Findings: outliers

The IQR rule flags values in every column, most often GGT (68), AST (61), BIL (46) and ALT (34); Age has only one. In total 166 of the 608 patients (27%) have at least one IQR outlier. These are concentrated in the disease groups: all 30 cirrhosis patients (100%), 17 of 21 fibrosis patients (81.0%) and 15 of 24 hepatitis patients (62.5%), compared with 104 of 533 healthy patients (19.5%). The Isolation Forest flags 31 rows, of which 28 are disease cases: 22 cirrhosis (73.3% of that group), 4 hepatitis (16.7%) and 2 fibrosis (9.5%), against 3 healthy patients (0.6%).

**No rows are removed.** Raised liver enzymes (AST, ALT, GGT) and bilirubin are expected features of liver disease, so the extreme values here are signal rather than measurement error. Removing them would delete most of the cirrhosis and fibrosis patients, who are already few (30 and 21), and would make the classes easier to separate than they are in practice. Outliers are reported only; no flags are added to the saved file. Models that are sensitive to scale are handled by standardisation inside the modelling pipeline.

## Save the cleaned data

The imputed dataset is saved without the dataframe index. No outlier flags are added to the file.

In [20]:
# Write the cleaned data. index=False avoids an extra index column.
# No outlier flags are added.
df_imputed.to_csv(OUTPUT_PATH, index=False)